Titanic NN Model


In [26]:
import pandas as pd
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import MinMaxScaler
import pickle

In [27]:
df = pd.read_csv("../data/titanic_prepared.csv")
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Embarked,Newfare
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,S,10.8750
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C,142.5666
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,S,11.8875
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,S,106.2000
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,S,12.0750


In [28]:
df.isna().sum()

PassengerId    0
Survived       0
Pclass         0
Name           0
Sex            0
Age            0
SibSp          0
Parch          0
Ticket         0
Fare           0
Embarked       0
Newfare        0
dtype: int64

In [29]:
df.dtypes

PassengerId      int64
Survived         int64
Pclass           int64
Name               str
Sex                str
Age            float64
SibSp            int64
Parch            int64
Ticket             str
Fare           float64
Embarked           str
Newfare        float64
dtype: object

In [30]:
df = df.drop(columns=["PassengerId", "Name", "Ticket", "Newfare"])
df.head()

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,0,3,male,22.0,1,0,7.2500,S
1,1,1,female,38.0,1,0,71.2833,C
2,1,3,female,26.0,0,0,7.9250,S
3,1,1,female,35.0,1,0,53.1000,S
4,0,3,male,35.0,0,0,8.0500,S


In [31]:
categorical_features = ["Sex", "Embarked"]


OneHot Encoder

In [32]:
one_hot_encoder = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
one_hot_encoder.fit(df[categorical_features])
one_hot_encoder

,categories,'auto'
,drop,None
,sparse_output,False
,dtype,<class 'numpy.float64'>
,handle_unknown,'ignore'
,min_frequency,None
,max_categories,None
,feature_name_combiner,'concat'


In [33]:
Column_features = one_hot_encoder.get_feature_names_out(categorical_features)
Column_features

array(['Sex_female', 'Sex_male', 'Embarked_C', 'Embarked_Q', 'Embarked_S'],
      dtype=object)

In [34]:
transformed_categorical_features = one_hot_encoder.transform(df[categorical_features])
transformed_categorical_features

array([[0., 1., 0., 0., 1.],
       [1., 0., 1., 0., 0.],
       [1., 0., 0., 0., 1.],
       ...,
       [1., 0., 0., 0., 1.],
       [0., 1., 1., 0., 0.],
       [0., 1., 0., 1., 0.]], shape=(881, 5))

Creating a Dataframe

In [35]:
df_transformed = pd.DataFrame(
    transformed_categorical_features, columns=Column_features)
df_transformed.head()

,Sex_female,Sex_male,Embarked_C,Embarked_Q,Embarked_S
0,0.0,1.0,0.0,0.0,1.0
1,1.0,0.0,1.0,0.0,0.0
2,1.0,0.0,0.0,0.0,1.0
3,1.0,0.0,0.0,0.0,1.0
4,0.0,1.0,0.0,0.0,1.0


Replacing the Dataframe with the categorical features


In [36]:
df = df.drop(columns=categorical_features)
df.head()

,Survived,Pclass,Age,SibSp,Parch,Fare
0,0,3,22.0,1,0,7.2500
1,1,1,38.0,1,0,71.2833
2,1,3,26.0,0,0,7.9250
3,1,1,35.0,1,0,53.1000
4,0,3,35.0,0,0,8.0500


In [37]:
final_df = df.join(df_transformed)
final_df.head()

,Survived,Pclass,Age,SibSp,Parch,Fare,Sex_female,Sex_male,Embarked_C,Embarked_Q,Embarked_S
0,0,3,22.0,1,0,7.2500,0.0,1.0,0.0,0.0,1.0
1,1,1,38.0,1,0,71.2833,1.0,0.0,1.0,0.0,0.0
2,1,3,26.0,0,0,7.9250,1.0,0.0,0.0,0.0,1.0
3,1,1,35.0,1,0,53.1000,1.0,0.0,0.0,0.0,1.0
4,0,3,35.0,0,0,8.0500,0.0,1.0,0.0,0.0,1.0


In [38]:
final_df.dtypes

Survived        int64
Pclass          int64
Age           float64
SibSp           int64
Parch           int64
Fare          float64
Sex_female    float64
Sex_male      float64
Embarked_C    float64
Embarked_Q    float64
Embarked_S    float64
dtype: object

In [39]:
final_df.describe()

,Survived,Pclass,Age,SibSp,Parch,Fare,Sex_female,Sex_male,Embarked_C,Embarked_Q,Embarked_S
count,881.000000,881.000000,881.000000,881.000000,881.00000,881.000000,881.000000,881.000000,881.000000,881.000000,881.000000
mean,0.384790,2.316686,29.153995,0.527809,0.38479,32.136511,0.354143,0.645857,0.188422,0.086266,0.725312
std,0.486822,0.833015,14.022600,1.107602,0.80943,49.877961,0.478524,0.478524,0.391271,0.280915,0.446610
min,0.000000,1.000000,0.420000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.000000,2.000000,20.000000,0.000000,0.00000,7.895800,0.000000,0.000000,0.000000,0.000000,0.000000
50%,0.000000,3.000000,28.000000,0.000000,0.00000,14.454200,0.000000,1.000000,0.000000,0.000000,1.000000
75%,1.000000,3.000000,38.000000,1.000000,0.00000,30.695800,1.000000,1.000000,0.000000,0.000000,1.000000
max,1.000000,3.000000,65.000000,8.000000,6.00000,512.329200,1.000000,1.000000,1.000000,1.000000,1.000000


MinMaxScaler

In [40]:
columns_to_scale = ["Age", "SibSp", "Parch", "Fare", "Pclass"]
scaler = MinMaxScaler(clip=True)
final_df[columns_to_scale] = scaler.fit_transform(final_df[columns_to_scale])
final_df.head()

,Survived,Pclass,Age,SibSp,Parch,Fare,Sex_female,Sex_male,Embarked_C,Embarked_Q,Embarked_S
0,0,1.0,0.334159,0.125,0.0,0.014151,0.0,1.0,0.0,0.0,1.0
1,1,0.0,0.581914,0.125,0.0,0.139136,1.0,0.0,1.0,0.0,0.0
2,1,1.0,0.396098,0.000,0.0,0.015469,1.0,0.0,0.0,0.0,1.0
3,1,0.0,0.535460,0.125,0.0,0.103644,1.0,0.0,0.0,0.0,1.0
4,0,1.0,0.535460,0.000,0.0,0.015713,0.0,1.0,0.0,0.0,1.0


In [41]:
final_df.describe()

,Survived,Pclass,Age,SibSp,Parch,Fare,Sex_female,Sex_male,Embarked_C,Embarked_Q,Embarked_S
count,881.000000,881.000000,881.000000,881.000000,881.000000,881.000000,881.000000,881.000000,881.000000,881.000000,881.000000
mean,0.384790,0.658343,0.444936,0.065976,0.064132,0.062726,0.354143,0.645857,0.188422,0.086266,0.725312
std,0.486822,0.416508,0.217135,0.138450,0.134905,0.097355,0.478524,0.478524,0.391271,0.280915,0.446610
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.000000,0.500000,0.303190,0.000000,0.000000,0.015412,0.000000,0.000000,0.000000,0.000000,0.000000
50%,0.000000,1.000000,0.427067,0.000000,0.000000,0.028213,0.000000,1.000000,0.000000,0.000000,1.000000
75%,1.000000,1.000000,0.581914,0.125000,0.000000,0.059914,1.000000,1.000000,0.000000,0.000000,1.000000
max,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


In [42]:
scaler.transform([[80, 0, 0, 300, 1]])

c:\Users\monat\anaconda3\envs\ai\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(


array([[1.      , 0.      , 0.      , 0.585561, 0.      ]])

Pickle

In [45]:
#Saving an object to disk
pickle.dump(scaler, open("../object/titanic_scaler.pkl", "wb"))

In [ ]:
#Loading an object from disk
scaler_loaded = pickle.load(open("../object/titanic_scaler.pkl", "rb"))